# Crawl bài báo khoa học từ PMC Open Access

Notebook này tải **tài liệu khoa học open-access** từ **PMC Open Access** bằng các endpoint chính thức của NCBI/PMC.


1. Tìm bài theo từ khóa bằng **NCBI E-utilities (ESearch)**
2. Lấy metadata cơ bản của PMCID bằng **ESummary**
3. Hỏi **PMC OA Web Service** để lấy link gói dữ liệu (`.tar.gz`) và PDF (nếu có)
4. Tải gói bài báo về máy, giải nén, và lưu `manifest.csv`

Mục tiêu của notebook là tạo một corpus có thể dùng tiếp cho benchmark dedup / parser-noise:
- `XML/NXML` = bản full text structured
- `PDF` = để parse bằng nhiều parser khác nhau sau này


## Ghi chú quan trọng

- Chỉ dùng **dịch vụ chính thức** của PMC/NCBI cho việc tải tự động.
- Chỉ tải các bài có trong **PMC Open Access Subset**.
- Với PDF, khả năng tải còn phụ thuộc loại license.
- Nếu bạn crawl số lượng lớn, hãy thêm email/API key và giữ nhịp request lịch sự.


In [1]:
!pip -q install requests pandas tqdm lxml

In [2]:
from __future__ import annotations

import io
import os
import re
import tarfile
import time
import json
from pathlib import Path
from typing import Dict, List, Optional
import xml.etree.ElementTree as ET

import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from tqdm.auto import tqdm


## Cấu hình

Bạn chỉ cần sửa `QUERY`, `MAX_ARTICLES`, và `OUT_DIR` là chạy được.

In [3]:
QUERY = 'large language model OR retrieval augmented generation'
MAX_ARTICLES = 10000
OUT_DIR = Path('pmc_oa_corpus')

EMAIL = ''
API_KEY = ''

REQUIRE_PDF = False

ONLY_PERMISSIVE_LICENSE = False

SLEEP_SECONDS = 0.35 if not API_KEY else 0.12

OUT_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / 'packages').mkdir(exist_ok=True)
(OUT_DIR / 'articles').mkdir(exist_ok=True)
(OUT_DIR / 'manifests').mkdir(exist_ok=True)

print('Output dir:', OUT_DIR.resolve())

Output dir: /kaggle/working/pmc_oa_corpus


## Helper functions

In [4]:
EUTILS_BASE = 'https://eutils.ncbi.nlm.nih.gov/entrez/eutils'
OA_BASE = 'https://www.ncbi.nlm.nih.gov/pmc/utils/oa/oa.fcgi'


def make_session() -> requests.Session:
    session = requests.Session()
    retries = Retry(
        total=5,
        backoff_factor=1.0,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=['GET'],
    )
    adapter = HTTPAdapter(max_retries=retries)
    session.mount('http://', adapter)
    session.mount('https://', adapter)
    session.headers.update({
        'User-Agent': 'pmc-oa-crawler/0.1 (contact: %s)' % (EMAIL or 'anonymous')
    })
    return session


session = make_session()


def ncbi_get(url: str, params: Dict) -> requests.Response:
    params = dict(params)
    if EMAIL:
        params['email'] = EMAIL
    if API_KEY:
        params['api_key'] = API_KEY
    resp = session.get(url, params=params, timeout=60)
    resp.raise_for_status()
    time.sleep(SLEEP_SECONDS)
    return resp


def ftp_to_https(url: str) -> str:
    return url.replace('ftp://ftp.ncbi.nlm.nih.gov/', 'https://ftp.ncbi.nlm.nih.gov/')


def normalize_pmcid(x: str) -> str:
    x = str(x).strip()
    return x if x.upper().startswith('PMC') else f'PMC{x}'


def is_permissive_license(license_text: str) -> bool:
    s = (license_text or '').upper()
    good = ['CC BY', 'CC0', 'CC BY-SA', 'CC BY-ND']
    return any(k in s for k in good)


## 1) Tìm PMCID theo từ khóa

In [5]:
def search_pmc(query: str, retmax: int = 100) -> List[str]:
    url = f'{EUTILS_BASE}/esearch.fcgi'
    params = {
        'db': 'pmc',
        'term': query,
        'retmax': retmax,
        'retmode': 'json',
        'sort': 'relevance',
    }
    data = ncbi_get(url, params).json()
    ids = data.get('esearchresult', {}).get('idlist', [])
    return [normalize_pmcid(x) for x in ids]


pmcids = search_pmc(QUERY, retmax=MAX_ARTICLES)
print('Found:', len(pmcids))
pmcids[:10]

Found: 9999


['PMC12933718',
 'PMC11211826',
 'PMC8287113',
 'PMC12637297',
 'PMC12947623',
 'PMC11837810',
 'PMC12157099',
 'PMC12953610',
 'PMC12998269',
 'PMC12080840']

## 2) Lấy metadata cơ bản

In [6]:
def chunked(items, size):
    for i in range(0, len(items), size):
        yield items[i:i+size]


def fetch_esummary(pmcids: List[str]) -> pd.DataFrame:
    rows = []
    url = f'{EUTILS_BASE}/esummary.fcgi'
    numeric_ids = [re.sub(r'^PMC', '', x, flags=re.I) for x in pmcids]

    for batch in tqdm(list(chunked(numeric_ids, 100)), desc='ESummary'):
        params = {
            'db': 'pmc',
            'id': ','.join(batch),
            'retmode': 'json',
        }
        data = ncbi_get(url, params).json()
        result = data.get('result', {})
        uids = result.get('uids', [])
        for uid in uids:
            rec = result.get(uid, {})
            article_ids = rec.get('articleids', []) or []
            by_type = {x.get('idtype'): x.get('value') for x in article_ids if isinstance(x, dict)}
            rows.append({
                'pmcid': normalize_pmcid(uid),
                'title': rec.get('title'),
                'pubdate': rec.get('pubdate'),
                'fulljournalname': rec.get('fulljournalname'),
                'authors': '; '.join(a.get('name', '') for a in rec.get('authors', []) if isinstance(a, dict)),
                'doi': by_type.get('doi'),
                'pmid': by_type.get('pmid'),
            })
    return pd.DataFrame(rows)


meta_df = fetch_esummary(pmcids)
meta_df.head()

ESummary:   0%|          | 0/100 [00:00<?, ?it/s]

,pmcid,title,pubdate,fulljournalname,authors,doi,pmid
0,PMC12933718,Large Language Model Agent for Modular Task Ex...,2026 Feb 23,Journal of chemical information and modeling,Ock J; Meda RS; Badrinarayanan S; Aluru NS; Ch...,10.1021/acs.jcim.5c02454,41662220
1,PMC11211826,Improving medical reasoning through retrieval ...,2024 Jun 28,"Bioinformatics (Oxford, England)",Jeong M; Sohn J; Sung M; Kang J,10.1093/bioinformatics/btae238,38940167
2,PMC8287113,Text Data Augmentation for Deep Learning.,2021,Journal of big data,Shorten C; Khoshgoftaar TM; Furht B,10.1186/s40537-021-00492-0,34306963
3,PMC12637297,Benchmarking retrieval-augmented large languag...,2025 Nov 21,Science advances,Li M; Zhan Z; Yang H; Xiao Y; Zhou H; Huang J;...,10.1126/sciadv.adr1443,41270181
4,PMC12947623,General-Purpose Models for the Chemical Scienc...,2026 Feb 25,Chemical reviews,Alampara N; Aneesh A; Ríos-García M; Mirza A; ...,10.1021/acs.chemrev.5c00583,41642594


## 3) Hỏi PMC OA service để lấy link package / PDF

OA service thường trả về:
- `tgz` package: gói chứa XML, media, supplementary và có thể có PDF
- `pdf`: link PDF riêng nếu license cho phép

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Dict, List
import pandas as pd
import xml.etree.ElementTree as ET
from tqdm.auto import tqdm

def fetch_oa_record(pmcid: str) -> Dict:
    resp = ncbi_get(OA_BASE, {'id': pmcid})
    root = ET.fromstring(resp.text)
    record = root.find('.//record')

    out = {
        'pmcid': pmcid,
        'citation': None,
        'license': None,
        'retracted': None,
        'tgz_href': None,
        'pdf_href': None,
        'links_raw': [],
        'oa_found': record is not None,
    }

    if record is None:
        return out

    out['citation'] = record.attrib.get('citation')
    out['license'] = record.attrib.get('license')
    out['retracted'] = record.attrib.get('retracted')

    for link in record.findall('link'):
        fmt = link.attrib.get('format')
        href = link.attrib.get('href')
        if href:
            href = ftp_to_https(href)

        out['links_raw'].append({
            'format': fmt,
            'href': href,
            **link.attrib
        })

        if fmt == 'tgz' and not out['tgz_href']:
            out['tgz_href'] = href
        if fmt == 'pdf' and not out['pdf_href']:
            out['pdf_href'] = href

    return out


def chunked(items, size):
    for i in range(0, len(items), size):
        yield items[i:i + size]


def fetch_oa_records_parallel(
    pmcids: List[str],
    batch_size: int = 100,
    max_workers: int = 8
) -> pd.DataFrame:
    rows = []

    for batch in tqdm(list(chunked(pmcids, batch_size)), desc="OA batches"):
        with ThreadPoolExecutor(max_workers=max_workers) as ex:
            futures = {ex.submit(fetch_oa_record, pmcid): pmcid for pmcid in batch}

            for fut in tqdm(
                as_completed(futures),
                total=len(futures),
                desc="Batch requests",
                leave=False
            ):
                pmcid = futures[fut]
                try:
                    rows.append(fut.result())
                except Exception as e:
                    rows.append({
                        'pmcid': pmcid,
                        'citation': None,
                        'license': None,
                        'retracted': None,
                        'tgz_href': None,
                        'pdf_href': None,
                        'links_raw': [],
                        'oa_found': False,
                        'error': repr(e),
                    })

    return pd.DataFrame(rows)


oa_df = fetch_oa_records_parallel(
    meta_df['pmcid'].tolist(),
    batch_size=100,
    max_workers=8
)

oa_df[['pmcid', 'license', 'tgz_href', 'pdf_href']].head()

OA batches:   0%|          | 0/100 [00:00<?, ?it/s]

Batch requests:   0%|          | 0/100 [00:00<?, ?it/s]

Batch requests:   0%|          | 0/100 [00:00<?, ?it/s]

Batch requests:   0%|          | 0/100 [00:00<?, ?it/s]

Batch requests:   0%|          | 0/100 [00:00<?, ?it/s]

...

Batch requests:   0%|          | 0/100 [00:00<?, ?it/s]

Batch requests:   0%|          | 0/100 [00:00<?, ?it/s]

Batch requests:   0%|          | 0/100 [00:00<?, ?it/s]

Batch requests:   0%|          | 0/100 [00:00<?, ?it/s]

Batch requests:   0%|          | 0/100 [00:00<?, ?it/s]

## 4) Ghép metadata + OA links, rồi lọc những bài tải được

In [ ]:
df = meta_df.merge(oa_df.drop(columns=['links_raw']), on='pmcid', how='left')

df['has_tgz'] = df['tgz_href'].notna()
df['has_pdf'] = df['pdf_href'].notna()
df['license_ok'] = df['license'].fillna('').map(is_permissive_license)

download_df = df[df['has_tgz']].copy()
if REQUIRE_PDF:
    download_df = download_df[download_df['has_pdf']]
if ONLY_PERMISSIVE_LICENSE:
    download_df = download_df[download_df['license_ok']]

print('Downloadable articles:', len(download_df))
download_df[['pmcid', 'title', 'license', 'has_pdf']].head(10)

## 5) Tải package `.tar.gz` và giải nén

In [ ]:
def safe_filename(text: str, max_len: int = 120) -> str:
    text = re.sub(r'[^\w\-\.]+', '_', text.strip())
    text = re.sub(r'_+', '_', text)
    return text[:max_len].strip('_') or 'article'


def download_file(url: str, path: Path) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and path.stat().st_size > 0:
        return path
    with session.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(path, 'wb') as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
    time.sleep(SLEEP_SECONDS)
    return path


def extract_tgz(tgz_path: Path, out_dir: Path) -> List[Path]:
    out_dir.mkdir(parents=True, exist_ok=True)
    extracted = []
    with tarfile.open(tgz_path, 'r:gz') as tar:
        members = tar.getmembers()
        tar.extractall(out_dir)
        for m in members:
            if m.isfile():
                extracted.append(out_dir / m.name)
    return extracted


def detect_main_files(article_dir: Path) -> Dict[str, Optional[str]]:
    xmls = sorted([p for p in article_dir.rglob('*') if p.suffix.lower() in {'.xml', '.nxml'}])
    pdfs = sorted(article_dir.rglob('*.pdf'))
    return {
        'xml_path': str(xmls[0]) if xmls else None,
        'pdf_path': str(pdfs[0]) if pdfs else None,
    }


In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Dict, List
import pandas as pd
from tqdm.auto import tqdm


def chunked(items, size):
    for i in range(0, len(items), size):
        yield items[i:i + size]


def process_one_article(row: Dict) -> Dict:
    pmcid = row['pmcid']
    tgz_url = row['tgz_href']
    article_slug = safe_filename(f"{pmcid}_{row.get('title') or ''}")

    tgz_path = OUT_DIR / 'packages' / f'{article_slug}.tar.gz'
    article_dir = OUT_DIR / 'articles' / article_slug

    try:
        download_file(tgz_url, tgz_path)
        extract_tgz(tgz_path, article_dir)
        detected = detect_main_files(article_dir)
        status = 'ok'
        error = None
    except Exception as e:
        detected = {'xml_path': None, 'pdf_path': None}
        status = 'error'
        error = repr(e)

    return {
        **row,
        'article_slug': article_slug,
        'tgz_path': str(tgz_path),
        'article_dir': str(article_dir),
        'status': status,
        'error': error,
        **detected,
    }


def download_and_extract_parallel(
    download_df: pd.DataFrame,
    batch_size: int = 50,
    max_workers: int = 8,
    save_every_batch: bool = True,
) -> pd.DataFrame:
    rows = []
    records = download_df.to_dict(orient='records')

    manifest_path = OUT_DIR / 'manifests' / 'manifest.csv'

    for batch_idx, batch in enumerate(
        tqdm(list(chunked(records, batch_size)), desc='Download batches'),
        start=1
    ):
        batch_rows = []

        with ThreadPoolExecutor(max_workers=max_workers) as ex:
            futures = {ex.submit(process_one_article, row): row['pmcid'] for row in batch}

            for fut in tqdm(
                as_completed(futures),
                total=len(futures),
                desc=f'Batch {batch_idx}',
                leave=False
            ):
                pmcid = futures[fut]
                try:
                    batch_rows.append(fut.result())
                except Exception as e:
                    row = next(x for x in batch if x['pmcid'] == pmcid)
                    batch_rows.append({
                        **row,
                        'article_slug': safe_filename(f"{pmcid}_{row.get('title') or ''}"),
                        'tgz_path': None,
                        'article_dir': None,
                        'status': 'error',
                        'error': repr(e),
                        'xml_path': None,
                        'pdf_path': None,
                    })

        rows.extend(batch_rows)

        if save_every_batch:
            pd.DataFrame(rows).to_csv(manifest_path, index=False)

    manifest_df = pd.DataFrame(rows)
    manifest_df.to_csv(manifest_path, index=False)
    return manifest_df


manifest_df = download_and_extract_parallel(
    download_df,
    batch_size=25,
    max_workers=8,
    save_every_batch=True,
)

manifest_df.head()

## 6) Kiểm tra nhanh những gì đã tải được

In [ ]:
summary = {
    'all_candidates': len(df),
    'downloadable_packages': int(df['has_tgz'].sum()),
    'downloadable_pdfs': int(df['has_pdf'].sum()),
    'download_ok': int((manifest_df['status'] == 'ok').sum()) if len(manifest_df) else 0,
    'xml_found': int(manifest_df['xml_path'].notna().sum()) if len(manifest_df) else 0,
    'pdf_found_after_extract': int(manifest_df['pdf_path'].notna().sum()) if len(manifest_df) else 0,
}
summary

## 7) Preview nội dung XML đầu tiên

In [ ]:
ok_xmls = manifest_df['xml_path'].dropna().tolist()
if ok_xmls:
    sample_xml = Path(ok_xmls[0])
    print('Sample XML:', sample_xml)
    text = sample_xml.read_text(encoding='utf-8', errors='ignore')
    print(text[:3000])
else:
    print('Chưa có XML nào được giải nén thành công.')

## 8) Bước tiếp theo cho benchmark dedup

Sau notebook này, bạn đã có:
- XML/NXML full text
- PDF của cùng bài (nếu package có hoặc nếu bạn tải riêng)

Bước tiếp theo để gần paper hơn:
1. Trích `official.txt` từ XML/NXML
2. Parse PDF bằng nhiều parser như PyMuPDF / Tesseract / Nougat
3. Gom các phiên bản đó theo cùng `pmcid`
4. Sinh positive pairs và hard negatives

Nếu muốn, mình có thể viết tiếp notebook thứ 2 để:
- parse XML thành `official.txt`
- parse PDF bằng nhiều parser
- xuất `clusters.parquet` cho benchmark dedup
